In [10]:
!pip install xgboost


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier

In [12]:
df = pd.read_csv("../data/Customer_Churn.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [13]:
df.drop("customerID", axis=1, inplace=True)

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df["TotalCharges"].fillna(
    df["TotalCharges"].median(),
    inplace=True
)

C:\Users\pooja\AppData\Local\Temp\ipykernel_7364\3246656213.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["TotalCharges"].fillna(


In [14]:
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

In [15]:
df["Monthly_to_Total"] = (
    df["MonthlyCharges"] /
    (df["TotalCharges"] + 1)
)

df["ChargePerMonth"] = (
    df["TotalCharges"] /
    (df["tenure"] + 1)
)

df["Tenure_Group"] = pd.cut(
    df["tenure"],
    bins=[0, 12, 24, 48, 72],
    labels=["0-1yr", "1-2yr", "2-4yr", "4-6yr"]
)

df["Tenure_Group"] = df["Tenure_Group"].fillna(
    df["Tenure_Group"].mode()[0]
)

df["SeniorCitizen"] = df["SeniorCitizen"].map({
    0: "No",
    1: "Yes"
})

In [16]:
df = pd.get_dummies(
    df,
    drop_first=True
)

In [17]:
X = df.drop(
    "Churn",
    axis=1
)

y = df["Churn"]

feature_names = X.columns.tolist()

print(X.shape)

(7043, 35)


In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [19]:
scaler = StandardScaler()

X_train = scaler.fit_transform(
    X_train
)

X_test = scaler.transform(
    X_test
)

In [20]:
smote = SMOTE(
    random_state=42
)

X_train_res, y_train_res = smote.fit_resample(
    X_train,
    y_train
)

print(X_train_res.shape)

(8278, 35)


In [21]:
param_grid = {
    "n_estimators": [200, 300],
    "max_depth": [4, 5, 6],
    "learning_rate": [0.03, 0.05],
    "subsample": [0.8, 0.9],
    "colsample_bytree": [0.8, 0.9]
}

grid = GridSearchCV(
    estimator=XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ),
    param_grid=param_grid,
    cv=3,
    scoring="f1",
    n_jobs=-1
)

grid.fit(
    X_train_res,
    y_train_res
)

print("Best Parameters:")
print(grid.best_params_)

Best Parameters:
{'colsample_bytree': 0.8, 'learning_rate': 0.03, 'max_depth': 4, 'n_estimators': 200, 'subsample': 0.9}


In [22]:
best_model = grid.best_estimator_

In [23]:
y_pred = best_model.predict(
    X_test
)

In [24]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred
)

recall = recall_score(
    y_test,
    y_pred
)

f1 = f1_score(
    y_test,
    y_pred
)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

Accuracy : 0.7750177430801988
Precision: 0.562363238512035
Recall   : 0.6871657754010695
F1 Score : 0.618531889290012


In [25]:
print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.88      0.81      0.84      1035
           1       0.56      0.69      0.62       374

    accuracy                           0.78      1409
   macro avg       0.72      0.75      0.73      1409
weighted avg       0.79      0.78      0.78      1409



In [26]:
cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)

[[835 200]
 [117 257]]


In [27]:
with open(
    "../models/xgboost.pkl",
    "wb"
) as f:
    pickle.dump(
        best_model,
        f
    )

In [28]:
with open(
    "../models/scaler.pkl",
    "wb"
) as f:
    pickle.dump(
        scaler,
        f
    )

In [29]:
with open(
    "../models/features.pkl",
    "wb"
) as f:
    pickle.dump(
        feature_names,
        f
    )

In [30]:
metrics = {
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1
}

with open(
    "../models/metrics.pkl",
    "wb"
) as f:
    pickle.dump(
        metrics,
        f
    )

In [31]:
with open(
    "../models/xgboost.pkl",
    "rb"
) as f:
    loaded_model = pickle.load(f)

print(
    loaded_model.predict(
        X_test[:5]
    )
)

[0 1 0 0 0]
